In [1]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 1.4 MB/s eta 0:00:00


In [2]:
# Célula 1
import numpy as np


class Ativacoes:
    @staticmethod
    def relu(Z):
        return np.maximum(0, Z)

    @staticmethod
    def relu_derivada(Z):
        return (Z > 0).astype(float)

    @staticmethod
    def softmax(Z):
        # Estabilidade numérica: subtrai o valor máximo de cada linha
        # Evita que o np.exp() estoure (overflow)
        Z_shift = Z - np.max(Z, axis=1, keepdims=True)
        exp_Z = np.exp(Z_shift)
        return exp_Z / np.sum(exp_Z, axis=1, keepdims=True)

class Perda:
    @staticmethod
    def cross_entropy(y_pred, y_true):
        n_amostras = y_true.shape[0]
        # Clip para evitar log(0) que resulta em NaN
        y_pred_clipped = np.clip(y_pred, 1e-12, 1.0 - 1e-12)
        return -np.sum(y_true * np.log(y_pred_clipped)) / n_amostras

In [3]:
# Célula 2
class CamadaDensa:
    def __init__(self, n_entradas, n_neuronios):
        # Inicialização de pesos (He Initialization - ideal para ReLU)
        self.pesos = np.random.randn(n_entradas, n_neuronios) * np.sqrt(2. / n_entradas)
        self.vieses = np.zeros((1, n_neuronios))
        
        # Estado do Momentum (v_t) inicializado em zero
        self.v_pesos = np.zeros_like(self.pesos)
        self.v_vieses = np.zeros_like(self.vieses)
        
        # Cache para o Backpropagation
        self.entrada = None
        self.Z = None

    def forward(self, entrada):
        self.entrada = entrada
        self.Z = np.dot(entrada, self.pesos) + self.vieses
        return self.Z

    def backward(self, dZ, taxa_aprendizado, momentum):
        n_amostras = self.entrada.shape[0]
        
        # Gradientes (Regra da Cadeia em formato matricial)
        d_pesos = np.dot(self.entrada.T, dZ) / n_amostras
        d_vieses = np.sum(dZ, axis=0, keepdims=True) / n_amostras
        d_entrada = np.dot(dZ, self.pesos.T)
        
        # Atualização com SGD + Momentum (vt = β * vt-1 + η * ∇W)
        self.v_pesos = (momentum * self.v_pesos) + (taxa_aprendizado * d_pesos)
        self.v_vieses = (momentum * self.v_vieses) + (taxa_aprendizado * d_vieses)
        
        self.pesos -= self.v_pesos
        self.vieses -= self.v_vieses
        
        return d_entrada

In [4]:
# Célula 3
class RedeNeuralMLP:
    def __init__(self):
        self.camadas = []

    def adicionar_camada(self, camada):
        self.camadas.append(camada)

    def forward(self, X):
        A = X
        # Passa por todas as camadas ocultas (usando ReLU)
        for camada in self.camadas[:-1]:
            Z = camada.forward(A)
            A = Ativacoes.relu(Z)
            
        # Última camada (usando Softmax)
        Z_saida = self.camadas[-1].forward(A)
        A_saida = Ativacoes.softmax(Z_saida)
        return A_saida

    def backward(self, X, y_true, y_pred, taxa_aprendizado=0.01, momentum=0.9):
        # A derivada da Cross Entropy combinada com Softmax é muito simples: (Predito - Real)
        dZ = y_pred - y_true
        
        # Derivada da última camada
        dA = self.camadas[-1].backward(dZ, taxa_aprendizado, momentum)
        
        # Backpropaga nas camadas ocultas, de trás para frente
        for camada in reversed(self.camadas[:-1]):
            # Regra da cadeia: Derivada da Ativação (ReLU) * Derivada que veio da frente
            dZ = dA * Ativacoes.relu_derivada(camada.Z)
            dA = camada.backward(dZ, taxa_aprendizado, momentum)

In [5]:
# Célula 4
from medmnist import PathMNIST
import numpy as np

# 1. Carregando a versão 28x28 apenas para a Etapa NumPy
print("Carregando dataset 28x28...")
train_dataset = PathMNIST(split="train", size=28, download=True)
val_dataset = PathMNIST(split="val", size=28, download=True)

# 2. Pré-processamento: Achatar (Flatten) e Normalizar (0 a 1)
# O shape original é (N, 28, 28, 3). Vamos transformar em (N, 28 * 28 * 3) = (N, 2352)
X_train = train_dataset.imgs.reshape(train_dataset.imgs.shape[0], -1) / 255.0
X_val = val_dataset.imgs.reshape(val_dataset.imgs.shape[0], -1) / 255.0

# 3. Pré-processamento: One-Hot Encoding para os rótulos
# Transforma o rótulo "2" em [0, 0, 1, 0, 0, 0, 0, 0, 0]
y_train_idx = train_dataset.labels.squeeze()
y_val_idx = val_dataset.labels.squeeze()

num_classes = 9
y_train = np.eye(num_classes)[y_train_idx]
y_val = np.eye(num_classes)[y_val_idx]

print(f"Formato de Entrada X: {X_train.shape}")
print(f"Formato dos Rótulos y: {y_train.shape}")

# 4. Instanciando a Rede Neural
modelo = RedeNeuralMLP()

# Camada Oculta: 2352 entradas -> 128 neurônios
modelo.adicionar_camada(CamadaDensa(X_train.shape[1], 128))
# Camada de Saída: 128 neurônios -> 9 neurônios (classes)
modelo.adicionar_camada(CamadaDensa(128, num_classes))

# 5. Loop de Treinamento (Epocas)
epocas = 15
taxa_aprendizado = 0.05
momentum = 0.9
batch_size = 256 # Processar em lotes para não estourar a memória

print("\nIniciando o treinamento do zero com NumPy...")

for epoca in range(epocas):
    # Embaralhar os dados a cada época
    indices = np.random.permutation(X_train.shape[0])
    X_train_shuffled = X_train[indices]
    y_train_shuffled = y_train[indices]
    
    loss_epoca = 0
    
    # Mini-batch Gradient Descent
    for i in range(0, X_train.shape[0], batch_size):
        X_batch = X_train_shuffled[i:i + batch_size]
        y_batch = y_train_shuffled[i:i + batch_size]
        
        # Forward pass
        predicoes = modelo.forward(X_batch)
        
        # Backward pass
        modelo.backward(X_batch, y_batch, predicoes, taxa_aprendizado, momentum)
        
        # Calcular perda do lote
        loss_epoca += Perda.cross_entropy(predicoes, y_batch)
    
    # Avaliação ao fim da época
    loss_media = loss_epoca / (X_train.shape[0] / batch_size)
    
    # Calcular acurácia na validação
    val_preds = modelo.forward(X_val)
    acertos = np.sum(np.argmax(val_preds, axis=1) == y_val_idx)
    acc_val = (acertos / len(y_val_idx)) * 100
    
    print(f"Época {epoca+1}/{epocas} - Loss: {loss_media:.4f} - Acurácia Validação: {acc_val:.2f}%")

Carregando dataset 28x28...


100%|██████████| 206M/206M [00:22<00:00, 9.00MB/s]


Formato de Entrada X: (89996, 2352)
Formato dos Rótulos y: (89996, 9)

Iniciando o treinamento do zero com NumPy...
Época 1/15 - Loss: 2.2090 - Acurácia Validação: 14.31%
Época 2/15 - Loss: 2.1881 - Acurácia Validação: 14.31%
Época 3/15 - Loss: 2.1879 - Acurácia Validação: 14.31%
Época 4/15 - Loss: 2.1881 - Acurácia Validação: 14.31%
Época 5/15 - Loss: 2.1881 - Acurácia Validação: 14.31%
Época 6/15 - Loss: 2.1880 - Acurácia Validação: 14.31%
Época 7/15 - Loss: 2.1880 - Acurácia Validação: 14.31%
Época 8/15 - Loss: 2.1879 - Acurácia Validação: 14.31%
Época 9/15 - Loss: 2.1880 - Acurácia Validação: 14.31%
Época 10/15 - Loss: 2.1880 - Acurácia Validação: 14.31%
Época 11/15 - Loss: 2.1880 - Acurácia Validação: 14.31%
Época 12/15 - Loss: 2.1374 - Acurácia Validação: 14.31%
Época 13/15 - Loss: 2.1880 - Acurácia Validação: 14.31%
Época 14/15 - Loss: 2.1878 - Acurácia Validação: 14.32%
Época 15/15 - Loss: 2.0872 - Acurácia Validação: 13.72%


In [6]:
# Célula 5
import numpy as np

def gradient_checking(modelo, X, y, epsilon=1e-7):
    print("--- Iniciando Gradient Checking na Última Camada ---")
    
    # Pega a última camada
    camada = modelo.camadas[-1]
    
    # 1. Forward e Backward normais para pegar o gradiente analítico
    y_pred = modelo.forward(X)
    dZ = y_pred - y
    n_amostras = X.shape[0]
    
    # Gradiente analítico
    grad_analitico = np.dot(camada.entrada.T, dZ) / n_amostras
    
    # 2. Gradiente Numérico (Força bruta)
    grad_numerico = np.zeros_like(camada.pesos)
    
    for i in range(5):
        for j in range(5):
            peso_original = camada.pesos[i, j]
            
            # f(W + epsilon)
            camada.pesos[i, j] = peso_original + epsilon
            loss_mais = Perda.cross_entropy(modelo.forward(X), y)
            
            # f(W - epsilon)
            camada.pesos[i, j] = peso_original - epsilon
            loss_menos = Perda.cross_entropy(modelo.forward(X), y)
            
            # Restaura
            camada.pesos[i, j] = peso_original
            
            # Aproximação
            grad_numerico[i, j] = (loss_mais - loss_menos) / (2 * epsilon)
            
    # 3. Diferença Relativa
    recorte_analitico = grad_analitico[:5, :5]
    recorte_numerico = grad_numerico[:5, :5]
    
    numerador = np.linalg.norm(recorte_analitico - recorte_numerico)
    # Adicionamos 1e-8 aqui para evitar divisão por zero (Erro NaN) em qualquer cenário
    denominador = np.linalg.norm(recorte_analitico) + np.linalg.norm(recorte_numerico) + 1e-8
    diferenca = numerador / denominador
    
    print(f"Diferença Relativa: {diferenca:.8e}")
    if diferenca < 1e-5:
        print("✅ SUCESSO! O Backpropagation está matematicamente correto!")
    else:
        print("❌ ALERTA! Possível erro no cálculo dos gradientes.")

# Pegamos uma pequena amostra dos dados
X_teste_grad = X_train[:5]
y_teste_grad = y_train[:5]

# INSTANCIAMOS UMA REDE NOVA E VIRGEM (Antes de qualquer treinamento)
modelo_teste = RedeNeuralMLP()
modelo_teste.adicionar_camada(CamadaDensa(X_train.shape[1], 128))
modelo_teste.adicionar_camada(CamadaDensa(128, 9))

# Forçamos um forward inicial para preencher as variáveis da camada
_ = modelo_teste.forward(X_teste_grad)

# Executamos o teste
gradient_checking(modelo_teste, X_teste_grad, y_teste_grad)

--- Iniciando Gradient Checking na Última Camada ---
Diferença Relativa: 1.05101490e-08
✅ SUCESSO! O Backpropagation está matematicamente correto!
